# 05 - Uçtan Uca Makine Öğrenmesi Hattı ve Keystone MLOps

> **Kapsam ve Hedef**: Scikit-Learn Pipeline mimarisi, model serileştirme (Joblib/Pickle), konfigürasyon yönetimi, çıkarım betiği ve üretime hazır yazılım mühendisliği pratikleri.

Bu notebook, **Keystone Mühendislik Prensipleri** doğrultusunda temiz kod, tip güvenliği ve veri bilimi odaklı bir yaklaşımla hazırlanmıştır.

---


### 1. `src/datascience_toolbox` Modüllerini İçe Aktarma
Keystone mimarisine göre notebook içinde tüm kodları baştan yazmak yerine, test edilmiş modüllerimizi kullanırız.


In [ ]:
import sys
from pathlib import Path

# Repo kök dizinini sys.path'e ekleme
sys.path.insert(0, "../../src")

from datascience_toolbox.data.loader import load_dataset
from datascience_toolbox.data.cleaner import DataCleaner
from datascience_toolbox.features.transformers import OutlierCapper
from datascience_toolbox.models.evaluator import evaluate_regression
from datascience_toolbox.utils.config import load_config

print("datascience_toolbox modülleri başarıyla içe aktarıldı!")


### 2. Konfigürasyon Dosyası ile Uçtan Uca Boru Hattı (Pipeline) Çalıştırma


In [ ]:
# 1. Konfigürasyonu yükle
cfg = load_config("../../configs/config.yaml")
print(f"Proje Adı: {cfg['project']['name']} | Versiyon: {cfg['project']['version']}")

# 2. Veriyi yükle
data_path = "../../" + cfg["data"]["housing_file"]
df_raw = load_dataset(data_path)

# 3. Temizle
cleaner = DataCleaner()
df_clean = cleaner.transform(df_raw)

# 4. Outlier Capping
capper = OutlierCapper(factor=1.5)
capper.fit(df_clean)
df_processed = capper.transform(df_clean)

print("İşlenmiş Veri Seti:")
print(df_processed.head(3))


### 3. Modeli Eğitme ve Performansını Değerlendirme


In [ ]:
X_mat = df_processed[["area_sqm", "rooms", "building_age", "distance_to_center_km"]].values
y_vec = df_processed["price"].values

# Basit doğrusal model ağırlıkları
weights, _, _, _ = np.linalg.lstsq(np.c_[np.ones(len(X_mat)), X_mat], y_vec, rcond=None)
y_predictions = np.c_[np.ones(len(X_mat)), X_mat].dot(weights)

# Model Değerlendirme Motoru
metrics = evaluate_regression(y_vec, y_predictions)
print("\nNihai Model Performans Raporu:")
for k, v in metrics.items():
    print(f" - {k.upper():<10}: {v}")
